In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
July = pd.read_csv('../July.csv', index_col=0)
July = July.iloc[::3]
July.reset_index(inplace=True, drop=True)

In [4]:
def interpolate(RH_in):
    kf1 = [1.37, 1.48, 1.726, 1.35, 1.78]
    kf2 = [1.71, 12.67,  33.39, 2.06, 14.54]
    T = [298, 299, 298, 308, 308]
    vol = [0.002, 0.01, 0.02, 0.01, 0.02]
    RH_list = []

    for i in range(len(T)):
        P_H2O_sat = np.exp(77.34 - 7235/T[i]-8.2*np.log(T[i]) + 0.005711*T[i])
        P_H2O = 1e5*vol[i]
        RH_list.append(P_H2O/P_H2O_sat*100)

    RH_kf = np.array([RH_list, kf1, kf2]).T
    RH_kf = RH_kf[RH_kf[:, 0].argsort()]

    kf1_hat = np.interp(RH_in, RH_kf[:, 0], RH_kf[:, 1])
    kf2_hat = np.interp(RH_in, RH_kf[:, 0], RH_kf[:, 2])

    return kf1_hat, kf2_hat

def H2O_vol(T_g, RH, P):
    P_H2O_sat = 0.61078*np.exp(17.269*(T_g-273.15)/(T_g-273.15+237.29))*1000
    P_H2O = P_H2O_sat*RH
    vol = P_H2O/P
    return vol

In [5]:
import subprocess
import os
import time
import sys
import logging
from threading import Timer
import psutil
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
import pandas as pd


def gPLOT2df(filename):
    """
    Convert .gPLOT file to a pandas dataframe

    Args:
        filename (str) : Name of the gPLOT file.

    Returns:
        df (DataFrame): Pandas dataframe with the time indexes on the rows and the variables on the columns.
    """
    with open(filename, 'r') as f:
        # Read the first line with the number of variables
        nvar = int(f.readline())
        # Read the variablenames
        vars = [f.readline().strip() for i in range(0, nvar)]
        # Read the variablevalues at each time point
        df = pd.read_csv(f, header=None, index_col=False)
        ntime = int((len(df) - 1) / (nvar + 1))
        # Reshape and drop termination element
        df = pd.DataFrame(data=df.values[0:-1].reshape((ntime, nvar + 1)), columns=['Time'] + vars)
    return df

In [7]:
def call_gORUN(gORUN_path, gENCRYPT_name, activity, process, password, working_dir='./', time_out=24*3600,
               verbose=True):
    """
    Calls gORUN and waits for the execution to complete.

    Args:
        gORUN (str): Path to gORUN.bat (must end with gORUN.bat)
        gENCRYPT_name (str): Name of the gENCRYPT file to execute
        activity (str): Activity type  ('sim'|'opt'|'est')
        process (str): Name of the process in the gPROMS file
        password (str): Encryption password of the process in the gPROMS file
        timeout (float): Time out in seconds. After this time the gO:RUN process will be terminated.
        working_dir (str): If the function is not called from the directory containing the input directory,
            use this to change to the right working directory first. The working directory will be set back to the
            original one before terminating.

    Returns:
        result (logical): Returns True for succes, False for error
    """
    # Start timer
    start_time = time.time()

    mpicmd_part_1 = "mpiexec.exe -n 1 -localonly"
    mpicmd_part_2 = "-mpmd"
    mpicmd_part_3 = ": -n NW gCluster.exe"

    # Construct gORUN command
    #gORUN_cmd = [gORUN_path, gENCRYPT_name, activity, process, password]
    gORUN_options =  gENCRYPT_name + " " +  activity + " " +  process + " " +  password
    gORUN_output = ' > ' + activity + '_output.txt'

    np=1
    if np > 1:
        gORUN_path = gORUN_path.rstrip('.bat') + '.exe'
        gORUN_cmd = mpicmd_part_1 + " \"" + gORUN_path + "\" " + mpicmd_part_2 + " " + gORUN_options + " " + \
                    mpicmd_part_3
    else:
        gORUN_cmd = gORUN_path + " " + gORUN_options + gORUN_output
    # Run gO:RUN and log output to file
#    try:
#        with open('./pygORUN.log', 'w') as f:
#     f = open('test.log', 'wb')

    #print(gORUN_cmd)
    print('gPROMS executed')
    process = subprocess.Popen(gORUN_cmd, cwd=working_dir,
                                       encoding='ascii')
#            for line in iter(process.stdout.readline, ""):
#                if verbose:
#                    sys.stdout.write(line)
#                f.write(line.rstrip('\n'))
#                f.flush()
#   except subprocess.CalledProcessError as e:
#        logging.error('Error calling gO:RUN')

    my_timer = Timer(time_out, lambda: kill_process(process))

    try:
        my_timer.start()
        output, error = process.communicate()
    finally:
        my_timer.cancel()

    elapsed_time = time.time() - start_time
    if elapsed_time > time_out:
        print("Activity timed out")


    if process.returncode == 0 and elapsed_time < time_out:
        return True
    else:
        return False

def kill_process(process_to_kill,timestamp="zzz"):
    # ----------------------------------------------------------------------------------------------
    # FEP new solution on killing gORUNs that hang
    try:
        parent = psutil.Process(process_to_kill.pid)
        children = parent.children(recursive=True)
        for child in children:
            child.kill()
        psutil.wait_procs(children, timeout=1)

        # having cleaned up all the children also clean up self
        process_to_kill.terminate()

    except:
        print("Unable to kill")
        pass

       

In [8]:
h2o_temp = np.zeros(July.shape[0])

for i in range(July.shape[0]):
    if July['tmpK'].iloc[i] <= 278:
        h2o_temp[i] = 278
    elif July['tmpK'].iloc[i] >= 303:
        h2o_temp[i] = 303
    else:
        h2o_temp[i] = July['tmpK'].iloc[i]

July['h2o_temperature'] = h2o_temp

kf1 = np.zeros(July.shape[0])
kf2 = np.zeros(July.shape[0])

for i in range(July.shape[0]):
    kf1[i], kf2[i] = interpolate([July['relh'].iloc[i]])

July['kf1, bar-1 s-1 (mol/kg)-1'] = kf1
July['kf2, bar2 s-1'] = kf2
July['pressure'] = 96890

July['H2O_vol'] = H2O_vol(July['tmpK'], July['relh']*0.01, July['pressure'])
July


,station,valid,tmpc,relh,sped,tmpK,index,YEAR,DOY,HOUR,RING,CO2,WIND,WIND_DIR,SOLAR_ANG,h2o_temperature,"kf1, bar-1 s-1 (mol/kg)-1","kf2, bar2 s-1",pressure,H2O_vol
0,OQT,2008-07-01 00:53:00,22.78,42.48,3.45,295.93,8395,2008,182,0,0,496.1,1.38,260,-30.77,295.93,1.767537,18.890409,96890,0.012154
1,OQT,2008-07-01 03:53:00,16.67,69.51,0.00,289.82,8410,2008,182,3,0,479.6,1.18,220,-18.63,289.82,1.726000,33.390000,96890,0.013613
2,OQT,2008-07-01 06:53:00,13.89,83.21,0.00,287.04,8425,2008,182,6,0,488.6,1.13,260,11.32,287.04,1.726000,33.390000,96890,0.013631
3,OQT,2008-07-01 09:53:00,12.22,89.95,0.00,285.37,8440,2008,182,9,0,375.6,2.01,240,47.09,285.37,1.726000,33.390000,96890,0.013210
4,OQT,2008-07-01 12:53:00,17.78,72.53,0.00,290.93,8455,2008,182,12,0,369.5,1.99,220,76.47,290.93,1.726000,33.390000,96890,0.015237
5,OQT,2008-07-01 15:53:00,25.56,37.38,5.75,298.71,8470,2008,182,15,0,365.8,3.32,260,51.45,298.71,1.777381,15.454184,96890,0.012634
6,OQT,2008-07-01 18:53:00,27.78,32.84,0.00,300.93,8485,2008,182,18,0,366.1,1.41,320,15.35,300.93,1.617310,13.525897,96890,0.012648
7,OQT,2008-07-01 21:53:00,28.89,26.55,5.75,302.04,8500,2008,182,21,0,429.2,0.35,240,-15.69,302.04,1.441419,9.521196,96890,0.010906
8,OQT,2008-07-02 00:53:00,25.00,38.74,0.00,298.15,8515,2008,183,0,0,483.6,0.37,0,-30.83,298.15,1.774756,16.370511,96890,0.012665
9,OQT,2008-07-02 03:53:00,18.33,70.28,0.00,291.48,8530,2008,183,3,0,525.5,0.29,260,-18.71,291.48,1.726000,33.390000,96890,0.015284


In [9]:
from PyDDSBB import DDSBB, DDSBBModel

In [10]:
t_opt = np.zeros([July.shape[0], 3])

for i in np.arange(20):
    sample = July.iloc[i]

    # A much faster way to run gPROMS
    def write_txt(cycle_time, No_Cycle, Ads0_time, input_file = 'inputs.txt'):
        # x are the independent input, K_sb is the adjustable
        parameter_sens = [sample['kf1, bar-1 s-1 (mol/kg)-1']*1e-5, sample['kf2, bar2 s-1']*1e-10]
        geo_inputs = [sample['H2O_vol'], sample['CO2']*1e-6, sample['tmpK'], sample['pressure'], sample['h2o_temperature']]
        f = open(input_file, "w+")
        f.write("Ads0_time"+"\n")
        f.write(str(np.round(Ads0_time))+"\n"+"\n")     

        f.write("Ads_time"+"\n")
        f.write(str(np.round(cycle_time[0]))+"\n"+"\n")

        f.write("Des_time"+"\n")
        f.write(str(np.round(cycle_time[1]))+"\n"+"\n")

        f.write("No_Cycle"+"\n")
        f.write(str(No_Cycle)+"\n"+"\n")

        f.write("kf_1"+"\n")
        f.write(str(parameter_sens[0])+"\n"+"\n")

        f.write("kf_2"+"\n")
        f.write(str(parameter_sens[1])+"\n"+"\n")

        f.write("y_H2O_0"+"\n")
        f.write(str(geo_inputs[0])+"\n"+"\n")

        f.write("y_CO2_0"+"\n")
        f.write(str(geo_inputs[1])+"\n"+"\n")

        f.write("T_feed"+"\n")
        f.write(str(geo_inputs[2])+"\n"+"\n")

        f.write("P_atm"+"\n")
        f.write(str(geo_inputs[3])+"\n"+"\n")

        f.write("T_h2o"+"\n")
        f.write(str(geo_inputs[4])+"\n"+"\n")

        #print(cycle_time)
        f.close()

    Dict = {}

    def sim_output(cycle_time):

        ads = np.round(cycle_time[0])
        Ads0 = ads*2
        vac1 = 5
        vac2 = 200
        vac3 = 100
        des = np.round(cycle_time[1])
        press1 = 20
        press2 = 200
        press3 = 5

        cycle = np.ceil(10000/(cycle_time[0]+cycle_time[1]))

        print(cycle_time, cycle)
        write_txt(cycle_time, No_Cycle = cycle, Ads0_time = Ads0, input_file = 'inputs.txt')

        gORUN_path = 'C:/Program Files/PSE/gPROMS-core_2021.2.0.55235/bin/goRUN.bat'
        gENCRYPT_name = 'H2O_export'
        activity = 'sim'
        process = 'H2O_export'
        password = 'H2O_export'

        call_gORUN(gORUN_path, gENCRYPT_name, activity, process, password, working_dir='./', time_out=24*3600,
                    verbose=True)

        filename = 'C:/Users/xcai65/OneDrive - Georgia Institute of Technology/study/research/NETL/script/Shubham/optimize_prod/output/H2O_export.gPLOT'
        output_results = gPLOT2df(filename)

        report_Des = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1
        report_Ads = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)
        report_Vac3 = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1-des-vac3

        try:
            #output_results['Time'] = np.round(output_results['Time'].values)
            prod_mass = output_results['Channel.PROD_MASS'][output_results['Time'] == report_Des].iloc[0] #mol/kg fiber/hr

            Out_CO2_des = output_results['Channel.OUT_CO2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
            Out_N2_des = output_results['Channel.OUT_N2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
            Out_H2O_des = output_results['Channel.OUT_H2O_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
            In_CO2_ads = output_results['Channel.IN_CO2_ads'][output_results['Time'] == report_Ads].iloc[0] #mol/channel

            Recovery = Out_CO2_des/In_CO2_ads
            Purity_CO2 = Out_CO2_des/(Out_CO2_des+Out_N2_des)

            E_blower = output_results['Channel.E_blower_ash'][output_results['Time'] == report_Ads].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
            E_vac = output_results['Channel.E_vac'][output_results['Time'] == report_Des].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
            Q_sens_ads = -(output_results['Channel.H_ads_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_ads_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
            Q_sens_CO2 = (output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
            Q_sens_H2O = (output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2    
            Q_ads_H2O = 50700*Out_H2O_des*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
            Q_ads_CO2 = (output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2  
            
            E_blower = E_blower*27.78
            E_vac = E_vac*(Out_CO2_des+Out_N2_des)/(Out_CO2_des+Out_N2_des+Out_H2O_des)

        except:
            prod_mass = 0
            Recovery = 0
            Out_CO2_des = 0
            Out_N2_des = 0
            Out_H2O_des = 0
            #Purity_H2O = 0
            E_blower = 0
            E_vac = 0
            Q_sens_ads = 0
            Q_sens_CO2 = 0
            Q_sens_H2O = 0
            Q_ads_H2O = 0
            Q_ads_CO2 = 0
        
        total_cost = (E_blower+E_vac)*0.05+(Q_sens_ads+Q_sens_CO2+Q_ads_CO2+Q_sens_H2O+Q_ads_H2O)*0.015


        if Recovery >= 0.2 and total_cost<=150:
             Dict[tuple(cycle_time)] = 1.
        else:
             Dict[tuple(cycle_time)] = 0.

        print(prod_mass, Recovery, total_cost)


        return prod_mass

    def simulator_cons(cycle_time):

         return Dict[tuple(cycle_time)]

    model = DDSBBModel.Problem()
    model.add_objective(sim_output, sense = 'maximize')

    model.add_variable(100., 2000.)
    model.add_variable(100., 2000.)
    model.add_unknown_constraint(simulator_cons)

    model_solver = DDSBB(20, split_method = 'equal_bisection', \
                            variable_selection = 'longest_side', multifidelity = 'nn', \
                            stop_option = {'absolute_tolerance': 0.05, 'relative_tolerance': 0.05, \
                                            'minimum_bound': 0.05, 'sampling_limit': 300, 'time_limit': 10000})


    model_solver.optimize(model)
    t_opt[i, 0] = model_solver.get_optimum()
    t_opt[i, 1:3] = model_solver.get_optimizer()

    print('Optimized cycle time for Month {} is {}'.format(i, t_opt[i, :]))

[100. 100.] 50.0
gPROMS executed
0 0 0.0
[2000. 2000.] 3.0
gPROMS executed
0.5125078319489333 0.3683956987567587 92.26358343919077
[ 106.43637606 1766.03850845] 6.0
gPROMS executed
0.1689008700139629 1.1938487896673593 212.99938913387666
[1653.96603406  182.86515047] 6.0
gPROMS executed
0.5949364543823007 0.23562150952140906 123.12693298010481
[1006.37999218 1012.94292708] 5.0
gPROMS executed
0.8505552597793449 0.6240942954604173 83.13842567387492
[ 997.90175227 1963.6439366 ] 4.0
gPROMS executed
0.5773006386948971 0.6268328537404784 83.28495262386933
[833.12092829 108.81592387] 11.0
gPROMS executed


KeyboardInterrupt: 

In [11]:
print(t_opt)

[[1.31687370e+00 1.78872604e+02 1.08792585e+02]
 [1.13502501e+00 4.23546492e+02 3.67802958e+02]
 [1.10046559e+00 6.37218284e+02 5.34790754e+02]
 [8.87999159e-01 1.08590883e+03 7.75780581e+02]
 [9.33051237e-01 7.46653290e+02 5.08263453e+02]
 [1.01056709e+00 3.00017968e+02 1.71904048e+02]
 [9.84733509e-01 2.51546017e+02 1.56184249e+02]
 [1.16768843e+00 1.66867914e+02 1.07072864e+02]
 [1.27939949e+00 1.60998259e+02 1.21254789e+02]
 [1.19082350e+00 5.12543378e+02 4.40965221e+02]
 [1.16885572e+00 5.25090643e+02 4.94549678e+02]
 [9.58353859e-01 8.28584471e+02 6.07691977e+02]
 [9.15669339e-01 7.80240397e+02 5.60323030e+02]
 [9.24407612e-01 4.58444860e+02 2.82730233e+02]
 [9.35658846e-01 2.94023740e+02 1.86385606e+02]
 [1.08573991e+00 1.96267518e+02 1.23910340e+02]
 [1.15964978e+00 3.68682237e+02 2.01206133e+02]
 [1.18709619e+00 5.80302359e+02 4.57631060e+02]
 [1.15886480e+00 4.90337506e+02 4.68815826e+02]
 [9.63926972e-01 7.28838500e+02 7.01997065e+02]
 [0.00000000e+00 0.00000000e+00 0.000000

In [12]:
df_out = pd.DataFrame(t_opt, columns=['Optimized Productivity, mol/kg/hr', 'Ads_t, s', 'Des_t, s'])

July = July.reset_index(drop=True)
July = pd.concat([July, df_out], axis = 1)
July


,station,valid,tmpc,relh,sped,tmpK,index,YEAR,DOY,HOUR,...,WIND_DIR,SOLAR_ANG,h2o_temperature,"kf1, bar-1 s-1 (mol/kg)-1","kf2, bar2 s-1",pressure,H2O_vol,"Optimized Productivity, mol/kg/hr","Ads_t, s","Des_t, s"
0,OQT,2008-07-01 00:53:00,22.78,42.48,3.45,295.93,8395,2008,182,0,...,260,-30.77,295.93,1.767537,18.890409,96890,0.012154,1.316874,178.872604,108.792585
1,OQT,2008-07-01 03:53:00,16.67,69.51,0.00,289.82,8410,2008,182,3,...,220,-18.63,289.82,1.726000,33.390000,96890,0.013613,1.135025,423.546492,367.802958
2,OQT,2008-07-01 06:53:00,13.89,83.21,0.00,287.04,8425,2008,182,6,...,260,11.32,287.04,1.726000,33.390000,96890,0.013631,1.100466,637.218284,534.790754
3,OQT,2008-07-01 09:53:00,12.22,89.95,0.00,285.37,8440,2008,182,9,...,240,47.09,285.37,1.726000,33.390000,96890,0.013210,0.887999,1085.908827,775.780581
4,OQT,2008-07-01 12:53:00,17.78,72.53,0.00,290.93,8455,2008,182,12,...,220,76.47,290.93,1.726000,33.390000,96890,0.015237,0.933051,746.653290,508.263453
5,OQT,2008-07-01 15:53:00,25.56,37.38,5.75,298.71,8470,2008,182,15,...,260,51.45,298.71,1.777381,15.454184,96890,0.012634,1.010567,300.017968,171.904048
6,OQT,2008-07-01 18:53:00,27.78,32.84,0.00,300.93,8485,2008,182,18,...,320,15.35,300.93,1.617310,13.525897,96890,0.012648,0.984734,251.546017,156.184249
7,OQT,2008-07-01 21:53:00,28.89,26.55,5.75,302.04,8500,2008,182,21,...,240,-15.69,302.04,1.441419,9.521196,96890,0.010906,1.167688,166.867914,107.072864
8,OQT,2008-07-02 00:53:00,25.00,38.74,0.00,298.15,8515,2008,183,0,...,0,-30.83,298.15,1.774756,16.370511,96890,0.012665,1.279399,160.998259,121.254789
9,OQT,2008-07-02 03:53:00,18.33,70.28,0.00,291.48,8530,2008,183,3,...,260,-18.71,291.48,1.726000,33.390000,96890,0.015284,1.190823,512.543378,440.965221


In [13]:
July.to_csv('July_prod_20.csv')

In [17]:
def write_txt(parameter_sens, geo_inputs, cycle_time, No_Cycle, Ads0_time, input_file = 'inputs.txt'):
    # x are the independent input, K_sb is the adjustable
    f = open(input_file, "w+")
    f.write("Ads0_time"+"\n")
    f.write(str(np.round(Ads0_time))+"\n"+"\n")     

    f.write("Ads_time"+"\n")
    f.write(str(np.round(cycle_time[0]))+"\n"+"\n")

    f.write("Des_time"+"\n")
    f.write(str(np.round(cycle_time[1]))+"\n"+"\n")

    f.write("No_Cycle"+"\n")
    f.write(str(No_Cycle)+"\n"+"\n")

    f.write("kf_1"+"\n")
    f.write(str(parameter_sens[0])+"\n"+"\n")

    f.write("kf_2"+"\n")
    f.write(str(parameter_sens[1])+"\n"+"\n")

    f.write("y_H2O_0"+"\n")
    f.write(str(geo_inputs[0])+"\n"+"\n")

    f.write("y_CO2_0"+"\n")
    f.write(str(geo_inputs[1])+"\n"+"\n")

    f.write("T_feed"+"\n")
    f.write(str(geo_inputs[2])+"\n"+"\n")

    f.write("P_atm"+"\n")
    f.write(str(geo_inputs[3])+"\n"+"\n")

    f.write("T_h2o"+"\n")
    f.write(str(geo_inputs[4])+"\n"+"\n")

    #print(cycle_time)
    f.close()
    
def sim_output(parameter_sens, geo_inputs, cycle_time):

    ads = np.round(cycle_time[0])
    Ads0 = ads*2
    vac1 = 5
    vac2 = 200
    vac3 = 100
    des = np.round(cycle_time[1])
    press1 = 20
    press2 = 200
    press3 = 5

    cycle = np.ceil(10000/(cycle_time[0]+cycle_time[1]))

    print(cycle_time, cycle)
    write_txt(parameter_sens, geo_inputs, cycle_time, No_Cycle = cycle, Ads0_time = Ads0, input_file = 'inputs.txt')


    gORUN_path = 'C:/Program Files/PSE/gPROMS-core_2021.2.0.55235/bin/goRUN.bat'
    gENCRYPT_name = 'H2O_export'
    activity = 'sim'
    process = 'H2O_export'
    password = 'H2O_export'

    call_gORUN(gORUN_path, gENCRYPT_name, activity, process, password, working_dir='./', time_out=24*3600,
                verbose=True)

    filename = 'C:/Users/xcai65/OneDrive - Georgia Institute of Technology/study/research/NETL/script/FACE/prod/output/H2O_export.gPLOT'
    output_results = gPLOT2df(filename)

    report_Des = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1
    report_Ads = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)
    report_Vac3 = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1-des-vac3

    try:
        #output_results['Time'] = np.round(output_results['Time'].values)
        prod_mass = output_results['Channel.PROD_MASS'][output_results['Time'] == report_Des].iloc[0] #mol/kg fiber/hr

        Out_CO2_des = output_results['Channel.OUT_CO2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
        Out_N2_des = output_results['Channel.OUT_N2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
        Out_H2O_des = output_results['Channel.OUT_H2O_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
        In_CO2_ads = output_results['Channel.IN_CO2_ads'][output_results['Time'] == report_Ads].iloc[0] #mol/channel

        Recovery = Out_CO2_des/In_CO2_ads
        Purity_CO2 = Out_CO2_des/(Out_CO2_des+Out_N2_des)
        #Purity_H2O = Out_H2O_des/(Out_CO2_des+Out_N2_des+Out_H2O_des)

        E_blower = output_results['Channel.E_blower_ash'][output_results['Time'] == report_Ads].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
        E_vac = output_results['Channel.E_vac'][output_results['Time'] == report_Des].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
        Q_sens_ads = -(output_results['Channel.H_ads_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_ads_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
        Q_sens_CO2 = (output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
        Q_sens_H2O = (output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2    
        Q_ads_H2O = 50700*Out_H2O_des*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
        Q_ads_CO2 = (output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2  

        E_blower = E_blower*27.78
        E_vac = E_vac*(Out_CO2_des+Out_N2_des)/(Out_CO2_des+Out_N2_des+Out_H2O_des)

    except:
        prod_mass = 0
        Recovery = 0
        Out_CO2_des = 0
        Out_N2_des = 0
        Out_H2O_des = 0
        #Purity_H2O = 0
        E_blower = 0
        E_vac = 0
        Q_sens_ads = 0
        Q_sens_CO2 = 0
        Q_sens_H2O = 0
        Q_ads_H2O = 0
        Q_ads_CO2 = 0



    return ([E_blower, E_vac, Q_sens_ads, Q_sens_CO2, Q_sens_H2O, Q_ads_CO2, Q_ads_H2O])


In [18]:
output_E = np.zeros([20, 7])

for i in range(July.shape[0]):
    sample = July.iloc[i, :]
    parameter_sens = [sample['kf1, bar-1 s-1 (mol/kg)-1']*1e-5, sample['kf2, bar2 s-1']*1e-10]
    geoinputs = [sample['H2O_vol'], sample['CO2']*1e-6, sample['tmpK'], sample['pressure'], sample['h2o_temperature']]
    cycle_time = [sample['Ads_t, s'], sample['Des_t, s']]

    output_E[i, :]= sim_output(parameter_sens, geoinputs, cycle_time)
    print(output_E[i, :],  i)

[178.87260424358078, 108.7925848980773] 35.0
gPROMS executed
[ 376.32745163  125.16243707 5314.06966308   20.49569787  392.52786127
  623.06754253 1599.41784033] 0
[423.54649204165634, 367.8029584101103] 13.0
gPROMS executed
[ 376.22252436  118.72075799 2708.70209625   50.31317528 1155.17450474
  635.45933531 2979.61726278] 1
[637.2182839647917, 534.7907536338267] 9.0
gPROMS executed
[ 393.95281861  116.60952327 2042.91749228   55.60178698 1336.66560214
  638.03111579 3329.28525458] 2
[1085.9088269268912, 775.7805812848616] 6.0
gPROMS executed
[ 523.71974522  115.30889255 1753.59900866   59.85535138 1532.02307405
  639.38748709 3745.64384446] 3
[746.6532901127848, 508.2634531750218] 8.0
gPROMS executed
[ 508.54545695  117.15534764 2169.70092275   56.85064944 1293.44591686
  639.76533917 3189.09977657] 4
[300.01796753305143, 171.9040483605013] 22.0
gPROMS executed
[ 501.20270795  124.72491889 4495.77362357   40.91982069  466.7874555
  627.97569623 1527.32016495] 5
[251.5460165067123, 15

C:\Users\xcai65\AppData\Local\Temp\ipykernel_4520\2180048781.py:52: RuntimeWarning: divide by zero encountered in double_scalars
  cycle = np.ceil(10000/(cycle_time[0]+cycle_time[1]))


IndexError: index 20 is out of bounds for axis 0 with size 20

In [ ]:
df_output_E = pd.DataFrame(output_E, columns = ['E_blower', 'E_vac', 'Q_sens_ads', 'Q_sens_CO2', 'Q_sens_H2O', 'Q_ads_CO2', 'Q_ads_H2O'])
df_output_E['Cost_E, $/tCO2'] = df_output_E.iloc[:, :2].sum(axis = 1)*0.05
df_output_E['Cost_H, $/tCO2'] = df_output_E.iloc[:, 2:7].sum(axis = 1)*0.015

df_final = pd.concat([July, df_output_E], axis = 1)
df_final.to_csv('July_prod_20.csv')